# 201 · Encode/decode cost playground

This notebook goes with the article
[Encode/decode cost](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/201/encode-decode-cost/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/201/encode_decode_cost.ipynb)

People sometimes say “JSON is slow, so switch to binary” as if that were a complete diagnosis.
Real cost usually comes from several places at once: finding structure in the bytes, converting numbers,
allocating objects, and copying buffers. This notebook makes those ideas visible with small experiments.

> **A note on numbers:** sizes and timings in these notebooks are only illustrations. For measured library comparisons on this project’s harness, use the suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) pages.


In [ ]:
import json
import struct
import timeit
from statistics import median

RECORD = {
    "id": 42,
    "temp_c": 21.5,
    "label": "sensor-7",
}



## Same logical value, different shapes on the wire

You encode one tiny sensor-like record three ways: compact JSON, a fixed little-endian binary layout (a toy, not a real standard),
and a small tagged binary sketch.

JSON should be the longest because it carries field names and decimal digits as text.
The binary forms should be shorter. Looking at the hexadecimal dump helps you see **where** the metadata and values live—
which is the first step toward reasoning about encode and decode cost.


In [ ]:
def hex_bytes(b: bytes) -> str:
    return " ".join(f"{x:02x}" for x in b)


json_bytes = json.dumps(RECORD, separators=(",", ":")).encode("utf-8")
# layout: id u32 LE, temp f64 LE, label len u8 + utf-8 (toy, not a real format)
label = RECORD["label"].encode("utf-8")
fixed = struct.pack("<IdB", RECORD["id"], RECORD["temp_c"], len(label)) + label


def encode_varint(u: int) -> bytes:
    out = bytearray()
    while u > 0x7F:
        out.append((u & 0x7F) | 0x80)
        u >>= 7
    out.append(u & 0x7F)
    return bytes(out)


# sketch: field tags as single bytes + varint id + f64 + len-string (still toy)
sketch = (
    b"\x01" + encode_varint(RECORD["id"])
    + b"\x02" + struct.pack("<d", RECORD["temp_c"])
    + b"\x03" + bytes([len(label)]) + label
)

rows = [
    ("JSON text", json_bytes),
    ("Fixed LE toy", fixed),
    ("Tagged binary sketch", sketch),
]
print(f"{'form':22} {'nbytes':>6}  hex (truncated)")
for name, b in rows:
    print(f"{name:22} {len(b):6}  {hex_bytes(b)[:60]}{'…' if len(b) > 20 else ''}")



## Decimal text versus binary floating-point bits

In JSON, the characters `2`, `1`, `.`, and `5` must be scanned and converted into a machine float.
In a binary layout, the same value can be stored as IEEE bits that the processor can load more directly.

This cell times many conversions each way.
Treat the ratio as a **teaching signal** only: it changes with hardware and Python version, and it is not a ranking of production libraries.


In [ ]:
def parse_json_number_many(n: int = 50_000) -> float:
    total = 0.0
    s = "21.5"
    for _ in range(n):
        total += float(s)  # stand-in for decimal conversion cost
    return total


def load_binary_float_many(n: int = 50_000) -> float:
    raw = struct.pack("<d", 21.5)
    total = 0.0
    for _ in range(n):
        total += struct.unpack("<d", raw)[0]
    return total


t_json = median(timeit.repeat(parse_json_number_many, number=1, repeat=5))
t_bin = median(timeit.repeat(load_binary_float_many, number=1, repeat=5))
print(f"median wall time for 50k conversions — decimal float(): {t_json*1e3:.2f} ms")
print(f"median wall time for 50k conversions — struct unpack:   {t_bin*1e3:.2f} ms")
print("Ratio (illustrative only):", round(t_json / t_bin, 2) if t_bin else "n/a")



## Payload shape can dominate the codec brand

Here you compare two JSON shapes with the same logical content: many small objects versus one denser record
that holds parallel lists. Expect the “many small objects” form to be larger and slower to round-trip,
because the parser creates more structure and allocates more.

Teams often change serializers and see little improvement when the object graph itself is still expensive.


In [ ]:
many_small = [{"k": i, "v": i * 0.1} for i in range(500)]
one_dense = {"ks": list(range(500)), "vs": [i * 0.1 for i in range(500)]}

b_many = json.dumps(many_small, separators=(",", ":")).encode()
b_one = json.dumps(one_dense, separators=(",", ":")).encode()
print("JSON size many small objects:", len(b_many))
print("JSON size one dense record:  ", len(b_one))


def roundtrip(data):
    return json.loads(json.dumps(data))


t_many = median(timeit.repeat(lambda: roundtrip(many_small), number=20, repeat=3))
t_one = median(timeit.repeat(lambda: roundtrip(one_dense), number=20, repeat=3))
print(f"round-trip median (20×) many-small: {t_many*1e3:.2f} ms")
print(f"round-trip median (20×) one-dense:  {t_one*1e3:.2f} ms")



## Takeaways

1. Think in cost centers—tokenizing, converting numbers, allocating, copying—not slogans.
2. A careful JSON implementation can beat a careless binary stack; payload shape matters either way.
3. To compare real libraries on this project’s hardware and harness, use the suite Results pages.

**Next:** [Self-describing vs schema](./self_describing_vs_schema.ipynb)
